# DP Audit Autoresearch — Autonomous Experiment Loop

**EECE 608 — Mohamad Faour**

Karpathy-style autonomous loop. An LLM (free via OpenRouter) proposes new
membership inference attacks, runs them, keeps improvements, reverts failures.

### One-time setup

1. kaggle.com/settings → Secrets → add `OPENROUTER_API_KEY` (free from openrouter.ai/keys)
2. Notebook sidebar: **Internet ON**, **Persistence: Files**, **CPU** accelerator
3. Notebook sidebar: Secrets → toggle ON `OPENROUTER_API_KEY`

---
## 0. Setup — clone repo, install deps

In [ ]:
import os, sys, shutil
from pathlib import Path

WORK = Path("/kaggle/working/EECE_608")

if not WORK.exists():
    !git clone https://github.com/mohdfaour03/EECE_608.git /kaggle/working/EECE_608
else:
    print(f"Already cloned: {WORK}")

os.chdir(WORK)
!pip install -q opacus>=1.5 dp-accounting>=0.4 pyyaml
!pip install -e . -q

sys.path.insert(0, str(WORK / "src"))
sys.path.insert(0, str(WORK / "autoresearch"))

print(f"\nReady.")

## 1. Resume from previous session (skip if first time)

If you saved state from a prior session (Cell 5), upload those files as a
dataset and uncomment below.

In [ ]:
AR = WORK / "autoresearch"

# === UNCOMMENT to restore previous session ===
# PREV = Path("/kaggle/input/autoresearch-state")  # change to your dataset
# for f in ["results.tsv", "approaches_log.jsonl", "experiment.py"]:
#     if (PREV / f).exists():
#         shutil.copy2(PREV / f, AR / f)
#         print(f"Restored {f}")

for f in ["results.tsv", "approaches_log.jsonl", "experiment.py"]:
    p = AR / f
    print(f"  {f}: {p.stat().st_size:,} bytes" if p.exists() else f"  {f}: fresh start")

## 2. Verify baseline

In [ ]:
import subprocess
r = subprocess.run(
    [sys.executable, str(AR / "experiment.py")],
    capture_output=True, text=True, timeout=180, cwd=str(WORK),
)
print(r.stdout)
if r.returncode != 0:
    print("STDERR:", r.stderr[-500:])
    raise RuntimeError("Baseline failed.")
print("Baseline OK.")

## 3. Run the loop

Leave this running. Each iteration: LLM proposes code → syntax check → run → keep/revert → log.

In [ ]:
from kaggle_secrets import UserSecretsClient

MODEL           = "qwen/qwen3-coder"
MAX_EXPERIMENTS = 150
TIMEOUT         = 300

api_key = UserSecretsClient().get_secret("OPENROUTER_API_KEY")

from agent_loop import run_loop

best = run_loop(
    api_key=api_key,
    model=MODEL,
    max_experiments=MAX_EXPERIMENTS,
    timeout=TIMEOUT,
)

## 4. Check progress (run anytime)

In [ ]:
import pandas as pd
tsv = AR / "results.tsv"
if tsv.exists():
    df = pd.read_csv(tsv, sep="\t")
    keeps = df[df["status"] == "keep"]
    print(f"Total: {len(df)} | Kept: {len(keeps)} | Best: {df['tightness_ratio'].max():.6f}\n")
    print(keeps.to_string(index=False))
    print(f"\nLast 5:")
    print(df.tail(5).to_string(index=False))
else:
    print("No results yet.")

## 5. Save state (before session expires)

Run this → Output tab → New Dataset → name `autoresearch-state`.
Next session: uncomment Cell 1 and point at that dataset.

In [ ]:
OUT = Path("/kaggle/working")
for f in ["results.tsv", "approaches_log.jsonl", "experiment.py"]:
    src = AR / f
    if src.exists():
        shutil.copy2(src, OUT / f)
        print(f"Saved {f} ({src.stat().st_size:,} bytes)")
print(f"\nCheck the Output tab.")

## 6. View best experiment code (optional)

In [ ]:
print((AR / "experiment.py").read_text())